In [ ]:
# Sample audio visualization
# Loads up to 4 sample audio files (one from different subfolders) and plots waveform + mel-spectrogram + MFCC
import matplotlib.pyplot as plt
import numpy as np

MAX_SAMPLES = 4

audio_sample_paths = samples[:MAX_SAMPLES]
if not audio_sample_paths:
    print('No sample audio files found to visualize.')
else:
    for i, p in enumerate(audio_sample_paths):
        print(f'[{i+1}] {p}')
        try:
            y, sr = librosa.load(p, sr=None)
        except Exception as e:
            print('  Could not load:', e)
            continue

        # Compute mel spectrogram and MFCC
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
        S_db = librosa.power_to_db(S, ref=np.max)
        mfcc = librosa.feature.mfcc(S=librosa.power_to_db(S), n_mfcc=13)

        fig, axes = plt.subplots(3, 1, figsize=(10, 7))
        times = np.arange(len(y)) / sr
        axes[0].plot(times, y, linewidth=0.6)
        axes[0].set_title(f'Waveform — {os.path.basename(p)} (sr={sr})')
        axes[0].set_xlim(0, times[-1])

        img = axes[1].imshow(S_db, origin='lower', aspect='auto', cmap='magma')
        axes[1].set_title('Mel Spectrogram (dB)')
        fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

        img2 = axes[2].imshow(mfcc, origin='lower', aspect='auto', cmap='viridis')
        axes[2].set_title('MFCC')
        fig.colorbar(img2, ax=axes[2])

        plt.tight_layout()
        plt.show()

# Project overview

This notebook is the canonical exploratory notebook for the project. It includes a concise overview of repository files, outputs, and quick run commands for the three tasks (easy, medium, hard).

**Quick smoke-run commands**
- Easy: `SAMPLE_SIZE=20 EPOCHS=1 python run_easy_task.py`
- Medium: `SAMPLE_SIZE=20 EPOCHS=1 python run_medium_task.py` (no genre usage by default; set `N_CLUSTERS` if needed)
- Hard: `RESOLVE_PATHS_ONLY=1 python run_hard_task.py` (checks file paths; set `SAMPLE_SIZE`/`EPOCHS` for a short run)

**Key files**
- `run_easy_task.py` — VAE on mel-spectrograms; t-SNE on latent; PCA + KMeans baseline
- `run_medium_task.py` — ConvVAE (audio + lyrics); KMeans/Agglomerative/DBSCAN on latent; PCA (hybrid) baseline
- `run_hard_task.py` — Beta-VAE with optional genre conditioning; saves reconstructions and baselines
- `src/` — dataset loaders, models, clustering helpers, evaluation and visualization utilities
- `results/` — metrics CSVs, indices CSVs, `latent_visualization/`, `reconstructions/hard/`, and `clusters_*.csv`

**Notes**
- Visuals default to **t-SNE** with adaptive perplexity.
- Medium task does **not** use genre labels for clustering by default.
- For full details see `README.md`.


# Exploratory Analysis: Unsupervised Clustering of Hybrid-Language Music
This notebook demonstrates data loading, feature extraction, training a simple Variational Autoencoder (VAE), clustering, and visualization for the hybrid-language music clustering project.

## Requirements
- pandas
- numpy
- librosa
- sentence-transformers
- torch
- scikit-learn
- matplotlib
- seaborn
- tqdm

Make sure to install the required packages before running the notebook.
Run this in your terminal:
`pip install pandas numpy librosa sentence-transformers torch scikit-learn matplotlib seaborn tqdm`

In [ ]:
# Quick utilities: preview README and list top-level files
import pathlib, os, glob
p = pathlib.Path('README.md')
print('README preview:\n')
if p.exists():
    print(p.read_text()[:800])
else:
    print('README.md not found in repo root')

print('\nTop-level files:')
for f in sorted(os.listdir('.')):
    print(f)

print('\nResults overview:')
for f in sorted(glob.glob('results/*'))[:20]:
    print(f)

# Preview key results files if present
import pandas as pd
metrics_files = [
    'results/clustering_metrics_easy.csv',
    'results/clustering_metrics_medium.csv',
    'results/clustering_metrics_hard.csv'
]
for mf in metrics_files:
    if os.path.exists(mf):
        print(f"\nPreview of {mf}:")
        print(pd.read_csv(mf).head(3))

In [ ]:
# Load and preview saved clustering metrics and indices (if present)
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = 'results'

metrics_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'clustering_metrics_*.csv')))
indices_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'clustering_indices_*.csv')))

print('Found metrics files:', metrics_files)
for mf in metrics_files:
    print('\nPreview:', os.path.basename(mf))
    try:
        df = pd.read_csv(mf)
        display(df.head())
    except Exception as e:
        print('  Could not read metrics file:', e)

print('\nFound indices files:', indices_files)
for inf in indices_files:
    print('\nPreview:', os.path.basename(inf))
    try:
        df = pd.read_csv(inf)
        display(df.head())
    except Exception as e:
        print('  Could not read indices file:', e)

# Show a quick summary table (concatenate if multiple)
if metrics_files:
    try:
        all_metrics = pd.concat([pd.read_csv(f).assign(source=os.path.basename(f)) for f in metrics_files], ignore_index=True)
        print('\nConcatenated metrics sample:')
        display(all_metrics.groupby('source').head(1))
    except Exception:
        pass

In [ ]:
# Display latent visualizations and reconstructions (if available)
import glob, os
import matplotlib.pyplot as plt

viz_root = os.path.join(RESULTS_DIR, 'latent_visualization')
print('Visualization root:', viz_root)

if os.path.exists(viz_root):
    pngs = sorted(glob.glob(os.path.join(viz_root, '*', '*.png')) + glob.glob(os.path.join(viz_root, '*.png')))
    print(f'Found {len(pngs)} visualization files')
    if pngs:
        preview = pngs[:4]
        fig, axes = plt.subplots(1, len(preview), figsize=(5*len(preview), 4))
        if len(preview) == 1:
            axes = [axes]
        for ax, p in zip(axes, preview):
            img = plt.imread(p)
            ax.imshow(img)
            ax.set_title(os.path.basename(p))
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('No latent_visualization dir found')

# Reconstructions (hard)
recon_dir = os.path.join(RESULTS_DIR, 'reconstructions', 'hard')
print('\nReconstructions dir:', recon_dir)
if os.path.exists(recon_dir):
    recon_files = sorted(glob.glob(os.path.join(recon_dir, '*.png')) + glob.glob(os.path.join(recon_dir, '*.jpg')))
    print(f'Found {len(recon_files)} reconstructions')
    if recon_files:
        preview = recon_files[:6]
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        for ax, p in zip(axes, preview):
            img = plt.imread(p)
            ax.imshow(img)
            ax.set_title(os.path.basename(p))
            ax.axis('off')
        for ax in axes[len(preview):]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('No reconstructions found for hard task')

## 1. Import libraries

In [ ]:
import os
import numpy as np
import librosa
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from src.dataset_medium import load_multimodal_features

## 2. Dataset Loading (Project paths)

This section shows how to discover and load multimodal features from the repository `data/` layout. We use `src.dataset_medium.load_multimodal_features` which can operate in two modes:

- `resolve_paths_only=True`: quickly checks that audio and lyrics files exist and returns paths (fast, no heavy feature extraction).
- `resolve_paths_only=False` (default): extracts mel-spectrograms and lyrics embeddings for each valid row; this can be time-consuming for the full dataset.

Example workflow below demonstrates both modes and how to create a small sample CSV (first 10 rows) for a fast smoke-run of feature extraction.

In [ ]:
# Inspect audio folder structure and resolve paths
import pandas as pd
import os
import glob
import librosa

CSV_PATH = 'data/metadata.csv'
AUDIO_DIR = 'data/audio'
LYRICS_DIR = 'data/lyrics'

print('Reading metadata...')
df = pd.read_csv(CSV_PATH)
print('Metadata shape:', df.shape)
print(df.head())

# Discover subfolders under data/audio (some datasets use numeric folders like 00_mp3, 01_mp3)
subfolders = [d for d in sorted(os.listdir(AUDIO_DIR)) if os.path.isdir(os.path.join(AUDIO_DIR, d))]
if not subfolders:
    # fallback: treat AUDIO_DIR as a flat folder
    subfolders = ['.']

print('\nAudio subfolders found:')
counts = {}
for s in subfolders:
    path = os.path.join(AUDIO_DIR, s) if s != '.' else AUDIO_DIR
    files = [f for f in os.listdir(path) if f.lower().endswith(('.wav', '.mp3'))]
    counts[s] = len(files)
    print(f'  {s:20s}: {len(files):5d} files')

# Show a few sample files (up to 6)
print('\nSample files (one from each subfolder):')
samples = []
for s in subfolders[:6]:
    path = os.path.join(AUDIO_DIR, s) if s != '.' else AUDIO_DIR
    files = [f for f in sorted(os.listdir(path)) if f.lower().endswith(('.wav', '.mp3'))]
    if files:
        samples.append(os.path.join(path, files[0]))
        print('-', os.path.join(path, files[0]))

print('\nResolving paths (fast check using dataset loader) ...')
audio_paths, lyrics_paths, df_res = load_multimodal_features(CSV_PATH, audio_base_dir=AUDIO_DIR, lyrics_base_dir=LYRICS_DIR, resolve_paths_only=True)
print(f'Found {len(audio_paths)} valid pairs (paths only).')

# Create a small sample CSV for quick feature extraction (first 10 rows)
sample_csv = 'data/_sample_metadata.csv'
df.head(10).to_csv(sample_csv, index=False)
print('\nRunning a small feature extraction on first 10 rows (sample) ...')
audio_feats, lyrics_feats, df_sample = load_multimodal_features(sample_csv, audio_base_dir=AUDIO_DIR, lyrics_base_dir=LYRICS_DIR, max_len=64)
print('Sample shapes — audio:', getattr(audio_feats, 'shape', None), 'lyrics:', getattr(lyrics_feats, 'shape', None))
print(df_sample.head())

# Cleanup (optional)
try:
    os.remove(sample_csv)
except Exception:
    pass

## 3. Variational Autoencoder (VAE) Model
Basic VAE architecture to encode audio and lyrics features into a latent space.

In [ ]:
# Note: project model implementations are available in `src/`. Prefer importing and using them for experiments.
# Example using the project's easy/medium models (replace class names as needed):
try:
    from src.vae_easy import VAE as VAE_easy
    print('Imported VAE from src.vae_easy')
except Exception:
    print('Project VAE not found in src.vae_easy; using the notebook Toy VAE')

# Toy VAE (kept for small demos)
class VAE(nn.Module):
    def __init__(self, audio_dim=(64, 216), lyrics_dim=384, latent_dim=32):
        super(VAE, self).__init__()
        self.audio_dim = audio_dim
        self.lyrics_dim = lyrics_dim
        self.latent_dim = latent_dim
        self.audio_flat_dim = audio_dim[0] * audio_dim[1]
        self.fc1 = nn.Linear(self.audio_flat_dim + lyrics_dim, 512)
        self.fc2_mu = nn.Linear(512, latent_dim)
        self.fc2_logvar = nn.Linear(512, latent_dim)
        self.fc3 = nn.Linear(latent_dim, 512)
        self.fc4 = nn.Linear(512, self.audio_flat_dim + lyrics_dim)
    def encode(self, x_audio, x_lyrics):
        x_audio = x_audio.view(-1, self.audio_flat_dim)
        x = torch.cat([x_audio, x_lyrics], dim=1)
        h1 = torch.relu(self.fc1(x))
        return self.fc2_mu(h1), self.fc2_logvar(h1)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z):
        h3 = torch.relu(self.fc3(z))
        recon = self.fc4(h3)
        return recon
    def forward(self, x_audio, x_lyrics):
        mu, logvar = self.encode(x_audio, x_lyrics)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

## 4. Loss Function and Training Loop

In [ ]:
def loss_function(recon_x, x, mu, logvar):
    BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    # KL divergence term
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

# Note: repository model classes include loss helpers and training loops; the following
# train_vae function is a small demo for the toy VAE above.

def train_vae(model, dataloader, epochs=20, lr=1e-3, device='cpu'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        train_loss = 0
        for mel_db, lyrics_emb in dataloader:
            mel_db = mel_db.to(device)
            lyrics_emb = lyrics_emb.to(device)
            optimizer.zero_grad()
            mel_db_flat = mel_db.view(mel_db.size(0), -1)
            inputs = torch.cat([mel_db_flat, lyrics_emb], dim=1)
            recon, mu, logvar = model(mel_db, lyrics_emb)
            loss = loss_function(recon, inputs, mu, logvar)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite loss encountered; reduce lr or check data')
            loss.backward()
            train_loss += loss.item()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {train_loss / len(dataloader.dataset):.4f}")
    return model

## 5. Extract Latent Representations

In [ ]:

def get_latent_representations(model, dataloader, device='cpu'):
    model.eval()
    latents = []
    with torch.no_grad():
        for mel_db, lyrics_emb in dataloader:
            mel_db = mel_db.to(device)
            lyrics_emb = lyrics_emb.to(device)
            mu, _ = model.encode(mel_db, lyrics_emb)
            latents.append(mu.cpu().numpy())
    return np.vstack(latents)


## 6. Clustering Latent Space

In [ ]:
def cluster_latent_space(latent_vectors, n_clusters=10):
    """Use KMeans for clustering (consistent with project scripts)."""
    scaler = StandardScaler()
    latent_scaled = scaler.fit_transform(latent_vectors)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans_labels = kmeans.fit_predict(latent_scaled)
    return kmeans_labels

## 7. Visualize Clusters with t-SNE

In [ ]:
def visualize_tsne(latent_vectors, labels=None, title='t-SNE projection'):
    """t-SNE visualization with adaptive perplexity (used across the project)."""
    n_samples = latent_vectors.shape[0]
    perplexity = min(30, max(5, n_samples // 3))
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    embedding = tsne.fit_transform(latent_vectors)

    plt.figure(figsize=(8,6))
    if labels is not None:
        scatter = plt.scatter(embedding[:,0], embedding[:,1], c=labels, cmap='tab10', s=10)
        plt.legend(*scatter.legend_elements(), title="Clusters")
    else:
        plt.scatter(embedding[:,0], embedding[:,1], s=10)
    plt.title(title)
    plt.xlabel('t-SNE 1')
    plt.ylabel('t-SNE 2')
    plt.show()

## 8. Usage Example 

In [ ]:

CSV_PATH = 'data/metadata.csv'
AUDIO_DIR = 'data/audio'
LYRICS_DIR = 'data/lyrics'

print(f"Using CSV: {CSV_PATH}")
print(f"Audio dir: {AUDIO_DIR}")
print(f"Lyrics dir: {LYRICS_DIR}")

# Load multimodal features from repo helper (use small SAMPLE_SIZE for quick runs)
audio_feats, lyrics_feats, df = load_multimodal_features(CSV_PATH, audio_base_dir=AUDIO_DIR, lyrics_base_dir=LYRICS_DIR, max_len=64)

if len(audio_feats) == 0:
    raise RuntimeError('No valid audio/lyrics pairs found. Check data paths and metadata.csv')

# audio_feats shape: (N, 1, n_mels, max_len) -> remove channel dim for toy VAE demo
if audio_feats.ndim == 4 and audio_feats.shape[1] == 1:
    audio_feats = np.squeeze(audio_feats, axis=1)  # now (N, n_mels, max_len)

# Convert to torch tensors and build DataLoader
audio_tensor = torch.tensor(audio_feats, dtype=torch.float32)
lyrics_tensor = torch.tensor(lyrics_feats, dtype=torch.float32)

dataset = TensorDataset(audio_tensor, lyrics_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Prepared dataset with {len(dataset)} samples, device={device}')

# Initialize and (optionally) train VAE - use small epochs for quick smoke tests
# Prefer the project model implementations in src (e.g., src.vae_easy.VAE or src.convae_medium.MultiModalConvVAE)
# For demo, if a project VAE is available, use it; otherwise fall back to the toy VAE defined in this notebook
try:
    from src.vae_easy import VAE as ProjectVAE
    print('Using Project VAE from src.vae_easy')
    vae_model = ProjectVAE()
except Exception:
    vae_model = VAE(audio_dim=(audio_tensor.shape[1], audio_tensor.shape[2]), lyrics_dim=lyrics_feats.shape[1], latent_dim=32)

trained_vae = train_vae(vae_model, dataloader, epochs=3, device=device)

# Extract latent vectors
latent_vectors = get_latent_representations(trained_vae, dataloader, device=device)

# Cluster latent space
kmeans_labels = cluster_latent_space(latent_vectors, n_clusters=10)

# Visualize clusters
visualize_tsne(latent_vectors, kmeans_labels, title='t-SNE of VAE Latent Space with K-Means Clusters')